In [ ]:
import json
import re
import os 
import pandas as pd

#Paths
#folder_base = r"C:\Users\ingri\Documents\IMIM\experiments\RJH006_subj_obj"
#experiment_name = "RJH006_subj_obj"
#json_path = f"{folder_base}/{experiment_name}_transcription.json"
csv_path = r"C:\Users\ingri\Documents\IMIM\experiments\RJH006_subj_obj\RJH006_subj_obj_names_timestamps_CSV_NEW.csv"
df = pd.read_csv(csv_path)

#Names to find
names = [
    {"name_chin": "朱茵", "name_eng": "Zhu Yin"},
    {"name_chin": "焦恩俊", "name_eng": "Jiao Enjun"},
    {"name_chin": "周星驰", "name_eng": "Stephen Chow"},
    {"name_chin": "金映娟", "name_eng": "Jin Yingjuan"},
    {"name_chin": "张怡宁", "name_eng": "Zhang Yining"},
]

#names_chin = []
ids_names = []

#loop through unique sentences

unique_sentences = list(df["english_sentence"].unique())
for i, sent in enumerate(unique_sentences):
    sentence_dicts = []
    total_count = 0
    speaker_id = df.loc[df["english_sentence"] == sent, ["speaker_id"]].values[0][0]

    #verifying names in sentence
    for name in names:
        n = name["name_eng"]
        if n in sent:
            id_name = sent.find(n)
            sentence_dicts.append({
                "sentence": sent,
                "name": n,
                "id_name": id_name
            })
            total_count += sent.count(n)
    
    #sentence type classification
    sent_lower = sent.lower()
    cond_list_names = (total_count == 5)
    cond_question = (("who" in sent_lower) and (total_count == 1)) and (speaker_id == 0)

    prev_sentence = unique_sentences[i-1].lower() if i > 0 else ""

    cond_answer = ("who" in prev_sentence) and (speaker_id == 1)
    #cond_answer = ((total_count < 4) and ( "who" in prev_sentence))
    cond_real_sentence = ((2 <= total_count <= 3) and (not cond_answer) and (not cond_question))
    if cond_list_names:
        type_sentence = "list_names"
    elif cond_question:
        type_sentence = "question" 
    elif cond_answer:
        type_sentence = "answer" 
    elif cond_real_sentence:
        type_sentence = "real_sentence"
    else:
        type_sentence = "error"
    

    #updating sentence dicts (type)
    for dic in sentence_dicts: 
        dic["type_sentence"] = type_sentence

    #list_ids = [ids["id_name"] for ids in ids_names]

        #classification subject/object
        if type_sentence == "real_sentence" and len(sentence_dicts) == 2: 
            n1, n2 = sentence_dicts
            if n1["id_name"] < n2["id_name"]: 
                n1["class_word"], n2["class_word"] = "subject", "object"
            else:
                n1["class_word"], n2["class_word"] = "object", "subject"

        else:
            for dic in sentence_dicts:
                dic["class_word"] = "none"

    #updating main ids_names dicts
    ids_names.extend(sentence_dicts)

#creating temporary auxiliary dataframe
ids_df = pd.DataFrame(ids_names)

#merging dataframes
new_df = df.merge(ids_df, how="left", left_on=["english_sentence", "english_name"], right_on=["sentence", "name"])

#dropping unnecessary columns
new_df = new_df.drop(columns=["sentence", "name", "id_name", "Unnamed: 0"], errors='ignore')

print(new_df.head(20))

#saving to csv and xlsx
output_csv_path = r"C:\Users\ingri\Documents\IMIM\experiments\RJH006_subj_obj" 
output_csv_file = "RJH006_subj_obj_names_timestamps_CLASSIFIED_CSV.csv" 
output_xlsx_file = "RJH006_subj_obj_names_timestamps_CLASSIFIED_XLSX_TEST_new.xlsx" 

#new_df.to_csv(f"{output_csv_path}/{output_csv_file}", index=False) 
new_df.to_excel(f"{output_csv_path}/{output_xlsx_file}", index=False) 

   chinese_name  english_name  start_time  end_time  \
0            朱茵       Zhu Yin        6.30      6.66   
1           焦恩俊    Jiao Enjun        8.16      8.76   
2           周星驰  Stephen Chow       10.86     11.72   
3           金映娟  Jin Yingjuan       13.44     14.10   
4           张怡宁  Zhang Yining       15.96     16.70   
5            朱茵       Zhu Yin       24.04     24.64   
6           焦恩俊    Jiao Enjun       25.10     25.74   
7           周星驰  Stephen Chow       27.30     27.96   
8           金映娟  Jin Yingjuan       26.40     27.10   
9           张怡宁  Zhang Yining       28.14     28.66   
10           朱茵       Zhu Yin       35.44     36.26   
11          周星驰  Stephen Chow       36.76     38.14   
12           朱茵       Zhu Yin       46.78     47.28   
13          周星驰  Stephen Chow       47.50     48.18   
14          焦恩俊    Jiao Enjun       53.44     54.08   
15          金映娟  Jin Yingjuan       51.94     52.78   
16          焦恩俊    Jiao Enjun       61.40     61.82   
17        

In [15]:
#Doing again, after fixing xlsx manualy
xlxs_path = r"C:\Users\ingri\Documents\IMIM\experiments\RJH006_subj_obj\RJH006_subj_obj_names_timestamps_CLASSIFIED_XLSX_TEST_new.xlsx"
df = pd.read_excel(xlxs_path)

#Names to find
names = [
    {"name_chin": "朱茵", "name_eng": "Zhu Yin"},
    {"name_chin": "焦恩俊", "name_eng": "Jiao Enjun"},
    {"name_chin": "周星驰", "name_eng": "Stephen Chow"},
    {"name_chin": "金映娟", "name_eng": "Jin Yingjuan"},
    {"name_chin": "张怡宁", "name_eng": "Zhang Yining"},
]

#names_chin = []
ids_names = []

#loop through unique sentences

unique_sentences = list(df["english_sentence"].unique())
for i, sent in enumerate(unique_sentences):
    sentence_dicts = []
    total_count = 0
    speaker_id = df.loc[df["english_sentence"] == sent, ["speaker_id"]].values[0][0]
    #print(speaker_id)

    #verifying names in sentence
    for name in names:
        n = name["name_eng"]
        if n in sent:
            id_name = sent.find(n)
            sentence_dicts.append({
                "sentence": sent,
                "name": n,
                "id_name": id_name
            })
            total_count += sent.count(n)
    
    #sentence type classification
    sent_lower = sent.lower()
    cond_list_names = total_count == 5
    cond_question = ("who" in sent_lower) and (total_count == 1) and (speaker_id == 0)

    prev_sentence = unique_sentences[i-1].lower() if i > 0 else ""

    cond_answer = ("who" in prev_sentence) and (speaker_id == 1)
    #cond_answer = ((total_count < 4) and ( "who" in prev_sentence))
    cond_real_sentence = (2 <= total_count <= 3) and (not cond_answer) and (not cond_question)
    if cond_list_names:
        type_sentence = "list_names"
    elif cond_question:
        type_sentence = "question" 
    elif cond_answer:
        type_sentence = "answer" 
    elif cond_real_sentence:
        type_sentence = "real_sentence"
    else:
        type_sentence = "error"
    

    #updating sentence dicts (type)
    for dic in sentence_dicts: 
        dic["type_sentence"] = type_sentence

    #list_ids = [ids["id_name"] for ids in ids_names]

        #classification subject/object
        if type_sentence == "real_sentence" and len(sentence_dicts) == 2: 
            n1, n2 = sentence_dicts
            if n1["id_name"] < n2["id_name"]: 
                n1["class_word"], n2["class_word"] = "subject", "object"
            else:
                n1["class_word"], n2["class_word"] = "object", "subject"

        else:
            for dic in sentence_dicts:
                dic["class_word"] = "none"

    #updating main ids_names dicts
    ids_names.extend(sentence_dicts)

#creating temporary auxiliary dataframe
ids_df = pd.DataFrame(ids_names)

#merging dataframes
new_df = df.merge(ids_df, how="left", left_on=["english_sentence", "english_name"], right_on=["sentence", "name"], suffixes=("", "_new"))

new_df["type_sentence"] = new_df["type_sentence_new"]
new_df["class_word"]    = new_df["class_word_new"]

#dropping unnecessary columns
new_df = new_df.drop(columns=["sentence", "name", "id_name", "Unnamed: 0", "type_sentence_new", "class_word_new", "type_sentence_new", "class_word_new"], errors='ignore')

print(new_df.head(20))

#saving to csv and xlsx
output_csv_path = r"C:\Users\ingri\Documents\IMIM\experiments\RJH006_subj_obj" 
output_csv_file = "RJH006_subj_obj_names_timestamps_CLASSIFIED_CSV.csv" 
output_xlsx_file = "RJH006_subj_obj_names_timestamps_CLASSIFIED_XLSX_TEST_new_toCHECK.xlsx" 

#new_df.to_csv(f"{output_csv_path}/{output_csv_file}", index=False) 
new_df.to_excel(f"{output_csv_path}/{output_xlsx_file}", index=False) 

   chinese_name  english_name  start_time  end_time  \
0            朱茵       Zhu Yin        6.30      6.66   
1           焦恩俊    Jiao Enjun        8.16      8.76   
2           周星驰  Stephen Chow       10.86     11.72   
3           金映娟  Jin Yingjuan       13.44     14.10   
4           张怡宁  Zhang Yining       15.96     16.70   
5            朱茵       Zhu Yin       24.04     24.64   
6           焦恩俊    Jiao Enjun       25.10     25.74   
7           周星驰  Stephen Chow       27.30     27.96   
8           金映娟  Jin Yingjuan       26.40     27.10   
9           张怡宁  Zhang Yining       28.14     28.66   
10           朱茵       Zhu Yin       35.44     36.26   
11          周星驰  Stephen Chow       36.76     38.14   
12           朱茵       Zhu Yin       46.78     47.28   
13          周星驰  Stephen Chow       47.50     48.18   
14          焦恩俊    Jiao Enjun       53.44     54.08   
15          金映娟  Jin Yingjuan       51.94     52.78   
16          焦恩俊    Jiao Enjun       61.40     61.82   
17        